# Time Scale Modification

In this homework you will implement several algorithms for time scale modification -- changing the playback speed of audio without changing the pitch.

As we are drawing near to the final project, I will be providing very little hand holding -- I will give you broad tasks to accomplish without providing answers. It is up to you to verify that your code is working!

Please note that an additional 10 points (10%) of your grade will be on the readability and decomposition of your code, as well as the clarity of your writing.  You should explain your thought process clearly and support your claims with appropriate plots.  Please submit your .ipynb and your generated audio recording (for the last part) as a .zip file.

In [1]:
%matplotlib inline

In [2]:
import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt
import librosa as lb
import IPython.display as ipd
from scipy.signal import medfilt

### Approach 1: Overlap-Add (OLA) [20 points]

The first approach you will implement is the overlap-add (OLA) method.  In this section, you should do the following:
- implement a function called tsm_overlap_add which takes three arguments: an input signal x, the time stretch factor alpha, and the frame length L.  It should return the time-stretched signal.  Your function should use default values of alpha=1 and L=220, and it should use a synthesis hop size that is half the value of L.
- apply your function to time stretch the beatboxing audio sample (with 22050 Hz sampling rate) by a factor of 1.25 (slowed down) and 0.75 (sped up).  Embed the resulting audio files in the notebook.
- apply your function to time stretch the choir audio sample by a factor of 1.25 (slowed down) and 0.75 (sped up).  Embed the resulting audio file in the notebook.
- Comment on the quality of the OLA algorithm when applied to the beatbox and choir audio samples.

In [ ]:
def hanning_window(L):
    """
    Returns a hann window of length L
    """
    return np.array(0.5 * (1 - np.cos(2 * np.pi / L * n)) for n in range(L))


In [ ]:
def tsm_overlap_add(x: NDArray[np.float64], alpha: float=1.0, L: int=220) -> NDArray[np.float64]:
    """
    Time-stretch signal x by factor alpha using overlap-add
    
    Parameters:
        x: input signal (1-dimensional numpy array)
        alpha: time stretch factor (float, default value = 1.0)
        L: frame length (int, default value = 220)
    Returns:
        y: time-stretched signal (1-dimensional numpy array)
    """
    # 1: Signal Decomposition
    # Goal: From original signal x, create analysis frames x_m
    # Set synthesis hop size H_s = L // 2
    # Calculate H_a = round(H_s / alpha)
    # Compute number of analysis frames: N_frames = (len(x) - L) // H_a + 1
    # Create analysis frames using following formula:
    # x_m [n] = x[n + m - H_a], n in range (0, N-1), 0 otherwise
    # Output: x_m as 1-dimensional array

    # 2: Frame mapping
    # Goal: Turn analysis frames into synthesis frames
    # Compute output length: output_length = (N_frames - 1) * H_s + L
    # Allocate output array y = zeros(out_len)
    # Create hann window length L by calling w = hann_window(L)
    w = hanning_window(L)
    # Create synthesis frames using following formula
    # y_m[n] = w[n] * x_m[n] / the sum on k of w[n - k * H_s]
    # Output: y_m as 1-dimensional array

    # 3: Signal Reconstruction
    # Goal: Create time-stretched signal y from synthesis frame array y_m
    # y[n] = the sum on k of y_m[n - k * H_s]
    # Output: y (final output of function)
    pass

### Approach 2: Phase Vocoder [45 points]

The second approach you will implement is the phase vocoder method.  In this section, you should do the following:
- implement a function called tsm_phase_vocoder which takes three arguments: an input signal x, the time stretch factor alpha, the frame length L, and the sample rate sr.  It should return the time-stretched signal.  Your function should use default values of alpha=1, L=2048, and sr=22050, and it should use a synthesis hop size that is one-fourth the value of L.  For the STFT, you may use the librosa stft function with FFT size 2048.  However, you should implement the istft function yourself (do *not* use the librosa istft function or look at its source code).
- apply your function to time stretch the beatboxing audio sample (with 22050 Hz sampling rate) by a factor of 1.25 (slowed down) and 0.75 (sped up).  Embed the resulting audio files in the notebook.
- apply your function to time stretch the choir audio sample by a factor of 1.25 (slowed down) and 0.75 (sped up).  Embed the resulting audio files in the notebook.
- Comment on the quality of the phase vocoder algorithm when applied to the beatbox and choir audio samples.

In [ ]:
def tsm_phase_vocoder(x: NDArray[np.float64], alpha: float=1.0, L: int=2048, sr: int=22050) -> NDArray[np.float64]:
    """
    Time-stretch signal x by factor alpha using phase vocoder algorithm

    Parameters:
        x: input signal (1-dimensional numpy array)
        alpha: time stretch factor (float, default value = 1.0)
        L: frame length (int, default value = 2048)
        sr: sampling rate in Hz(int, default value = 22050)
    Returns:
        y: time-stretched signal (1-dimensional numpy array)
    """
    # 1: Compute STFT
    # Goal: compute windowed STFT on analysis frames
    # select Ha = Hs/alpha 
    # Compute windowed STFT on analysis frames (can use librosa)
    # stack into matrix
    # Output: STFT matrix X
    
    # 2: Modify STFT
    # Goal: Modify STFT matrix X to keep the magnitudes the same, modify phases to the following:
    # X_mod (m, k) = |X(m,k)| * e^{j * phi_mod (m,k)}
    # phi_mod (0,k) = phi(0,k)
    # where phi_mod (m+1, k) = phi_mod(m,k) + w(m,k) * H_s/f_s, calculated recursively
    # w(m,k) = w_nom(k) + Phi(phi(n+1,k) - phi(m,k) - w_nom(k)*(delta t))/(delta t)

    # 3: Signal reconstruction
    # Goal: Reconstruct signal y whose STFT is close to X_mod
    # Compute inverse fft on each frame (call my_istft())
    # derive synthesis frames y_m using following formula:
    # y_m[n] = w[n] * x_m^Mod[n] / the sum on k of w[n-k*H_s]^2
    # reconstruct time signal y using the following formula:
    # y[n] = the sum on k of y_m[n - k*H_s]
    # Output: time-domain signal y

    pass

In [ ]:
def my_istft(S, hop_length, window_length):
    """
    Manual implementation of iSTFT

    Parameters:
        S: complex STFT matrix shape (n_fft//2 + 1, n_frames)
        hop_length: hop size in samples
        window_length: STFT window size
    """
    n_freq_bins = S.shape[0]
    n_frames = S.shape[1]
    n_bins = 2*(n_freq_bins-1)
    signal_length = (n_frames - 1) * hop_length + window_length
    
    # initialize output
    y = np.zeros(signal_length)
    window_sum = np.zeros(signal_length)

    window = hanning_window(window_length)
    for i in range(n_frames):
        # 1. Take the Inverse Real FFT of the current column
        # This converts frequency domain back to time domain
        frame_signal = np.fft.irfft(S[:, i], n=n_bins)
        
        # 2. Apply the synthesis window
        # (This handles the transition between overlapping frames)
        frame_windowed = frame_signal * window
        
        # 3. Overlap-Add into the output buffer
        start = i * hop_length
        end = start + window_length
        
        # Ensure we don't go out of bounds if the window is longer than n_fft
        y[start:end] += frame_windowed
        window_sum[start:end] += window ** 2

    # 4. Normalize by the window energy to ensure perfect reconstruction
    # We avoid division by zero where windows don't overlap
    approx_nonzero_indices = window_sum > 1e-10
    y[approx_nonzero_indices] /= window_sum[approx_nonzero_indices]

    return y

### Harmonic Percussive Separation [15 points]

In preparation for implementing the third time scale modification approach, you will implement harmonic percussive separation.  In this section, you should do the following:
- implement a function called harmonic_percussive_separation which takes five arguments: an input signal x, sample rate sr, FFT size fft_size, hop size hop_length, harmonic median filter half-length lh, and percussive median filter half-length lp.  It should return four outputs: the two time-domain signals xh and xp corresponding to the harmonic and percussive components, and the two corresponding modified STFTs Xh and Xp.  Your function should use default values sr=22050, fft_size=2048, hop_length=512, lh=6, and lp=6.  You may use the librosa stft function, but you should implement the rest yourself.  You should *not* use the librosa hpss function or look at its source code.  Make sure to reuse your istft function from above!
- superimpose the choir and beatboxing audio recordings together by taking the average of their time domain signals.  Process the mixture with your algorithm to estimate the harmonic and percussive components, and embed the two resulting audio recordings in the notebook.
- Compute the log-magnitude spectrogram of the estimated xh and xp signals, and visualize a short section of both specgtrograms.  Include the images in your notebook and comment on what you see.

In [ ]:
def harmonic_percussive_separation(x: NDArray[np.float64],
                                   sr: int=22050,
                                   fft_size: int=2048,
                                   hop_length: int=512,
                                   lh: int=6,
                                   lp: int=6):
    """
    Separate signal x into harmonic and percussive components

    Parameters:
        x: input signal (1-dimensional numpy array)
        sr: sampling rate (default 22050)
        fft_size: FFT bin size (default 2048)
        hop_length: STFT hop size (default 512)
        lh: half-length of harmonic median filter, applied along time axis (default 6)
        lp: half-length of percussive median filter, applied along frequency axis (default 6)
    
    Returns:
        xh: harmonic component (time domain)
        xp: percussive component (time domain)
        Xh: harmonic component (frequency domain)
        Xp: percussive component (frequency domain)
    """

    # 1: Calculate STFT X(m,k)
    # Return STFT matrix X (M x K)
    X = lb.stft(n_fft = fft_size, hop_length = hop_length)
    M = X.shape[0]
    K = X.shape[1]
    # 2: Filter magnitude spectrogram
    # Goal: Create two separate magnitude spectrograms for percussive & harmonic components
    Y = np.abs(X)
    # Create magnitude spectrogram Y(m,k) = abs(X(m,k))
    # Harmonic spectrogram: Yh(m,k) = median[Y(m-lh,k),...,Y(m+lh,k)]
    Yh = np.stack([medfilt(Y[k,:], kernel_size=2*lh+1) for k in range(M)], axis=0)
    # Percussive spectrogram: Yo(m,k) = median[Y(m,k-lp),...,Y(m,k+lp)]
    Yp = np.stack([medfilt(Y[:,k], kernel_size=2*lp+1) for k in range(K)], axis=1)
    # Output: Yh and Yp

    # 3: Generate binary masks
    # Goal: Create two separate binary masks for percussive and harmonic components
    # Mh(m,k) = 1 if Yh(m,k) > Yp(m,k), 0 otherwise
    # Mp(m,k) = 1 if Yp(m,k) > Yh(m,k), 0 otherwise
    # Output: Binary masks Mh and Mp
    
    # 4: Apply masks
    # Goal: Apply masks to magnitude spectrogram
    # Xh = X .* Mh
    # Xp = X .* Mp
    # Output: Masked percussive and harmonic STFT matrices

    # 5: Reconstruct signal (same as phase vocoder step 3)
    # Can't we just use ifft()?
    #
    # From phase vocoder
    # Goal: Reconstruct signal y whose STFT is close to X_mod
    # Compute inverse fft on each frame (call my_istft())
    # derive synthesis frames y_m using following formula:
    # y_m[n] = w[n] * x_m^Mod[n] / the sum on k of w[n-k*H_s]^2
    # reconstruct time signal y using the following formula:
    # y[n] = the sum on k of y_m[n - k*H_s]

    # Output: time-domain signal xh and xp

    # To verify it works, add xh and xp. should be the same as x
    pass
    return xh, xp, Xh, xp

### Approach 3: Hybrid Method [5 points]

The third approach you will implement is the hybrid method that combines both the overlap-add and phase vocoder methods. In this section, you should do the following:
- implement a function called tsm_hybrid which takes five arguments: an input signal x, time stretch factor alpha, and sample rate sr.  It should return the time-stretched signal y.  Your function should assume default values of alpha=1, sr=22050.  You should call the functions you implemented in earlier sections and use the corresponding default values.
- apply your function to time stretch the beatboxing audio sample by a factor of 1.25 (slowed down) and 0.75 (sped up). Embed the resulting audio files in the notebook.
- apply your function to time stretch the choir audio sample by a factor of 1.25 (slowed down) and 0.75 (sped up). Embed the resulting audio files in the notebook.
- Comment on the quality of the hybrid algorithm when applied to the beatbox and choir audio samples.

In [ ]:
def tsm_hybrid(x: NDArray[np.float64], alpha: float=1.0, sr: int=22050) -> NDArray[np.float64]:
    """
    Time-stretch signal x by factor alpha by combining overlap-add and phase vocoder methods

    Parameters:
        x: input signal (1-dimensional numpy array)
        alpha: time stretch factor (default 1.0)
        sr: sampling rate in Hz (default 22050)
    
    Returns:
        y: time-stretched signal (1-dimensional numpy array)
    """
    # Separate x into xp and xh by calling harmonic_percussive_separation
    xh, xp, _, _ = harmonic_percussive_separation(x, sr=sr)

    # Time-stretch harmonic component with phase vocoder
    yh = tsm_phase_vocoder(xh, alpha=alpha, sr=sr)

    # Time-stretch percussive component with OLA
    yp = tsm_overlap_add(xp, alpha=alpha)

    # Match lengths by trimming longer one if necessary
    min_len = min(len(yh), len(yp))
    y = yh[:min_len] + yp[:min_len]
    
    return y

### Make something cool! [5 points]

Now use the functions you've implemented above to create a cool 10-15 second audio sample of any music you like.  Feel free to time stretch, pitch shift, and mix audio recordings to create something original.  You should include the following:
- code for generating the audio sample
- the audio file embedded into the notebook
- a brief explanation of what you did

Please also include your audio file in the zip file with your submission.